# 📈 Time Series Analysis — Concepts Notebook
### Sample Data se Trend, Seasonality aur Forecasting Samjho

---

**Topics Covered:**
1. 🕐 Introduction to Time Series
2. 📊 Components: Trend, Seasonal, Cyclical, Irregular
3. 🔬 Stationarity aur ADF Test
4. 📉 ACF aur PACF Plots
5. 🔄 AR, MA, ARMA Models
6. 📐 ARIMA aur SARIMA
7. 🏆 Capstone + Solutions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('✅ Ready!')

---
## 🕐 Section 1: Introduction to Time Series

**Naye Keywords:**
- **Time Series** — time ke saath collect kiya gaya data (har point ek time stamp ke saath)
- **Univariate** — sirf ek variable track karna (e.g., daily temperature)
- **Multivariate** — multiple variables ek saath (e.g., temp + humidity + pressure)
- **Lag** — pichle time steps ke values use karna
- **Forecast Horizon** — kitne future steps predict karne hain

In [ ]:
# --- Generate a realistic synthetic time series ---
np.random.seed(42)
n = 120  # 10 years monthly data
t = np.arange(n)

trend = 0.5 * t                              # Linear trend
seasonal = 20 * np.sin(2 * np.pi * t / 12)  # Yearly seasonality
noise = np.random.normal(0, 5, n)            # Random noise
series = 100 + trend + seasonal + noise

dates = pd.date_range(start='2015-01', periods=n, freq='MS')
ts = pd.Series(series, index=dates, name='Sales')

plt.figure(figsize=(12, 4))
plt.plot(ts, color='steelblue', linewidth=1.5)
plt.title('Monthly Sales Data (2015-2024)', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Time Series Summary:")
print(ts.describe().round(2))

---
## 📊 Section 2: Components of Time Series

**Naye Keywords:**
- **Trend (T)** — long-term upward/downward movement
- **Seasonality (S)** — fixed period repeating pattern (monthly, yearly)
- **Cyclical (C)** — longer irregular cycles (business cycles)
- **Irregular/Residual (I)** — random noise, unexplained variation
- **Additive Model** — Y = T + S + C + I (components add up)
- **Multiplicative Model** — Y = T × S × C × I (percentage effect)

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Decompose the time series
decomp = seasonal_decompose(ts, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10))
components = [
    (ts, 'Original', 'steelblue'),
    (decomp.trend, 'Trend', 'darkorange'),
    (decomp.seasonal, 'Seasonal', 'green'),
    (decomp.resid, 'Residual (Noise)', 'red')
]
for ax, (comp, title, color) in zip(axes, components):
    ax.plot(comp, color=color, linewidth=1.5)
    ax.set_title(title)
    ax.grid(alpha=0.3)

plt.suptitle('Time Series Decomposition (Additive)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Manually show components ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

t_arr = np.arange(48)
axes[0].plot(0.5 * t_arr, color='darkorange', linewidth=2)
axes[0].set_title('Trend (Linear Upward)')
axes[0].set_xlabel('Time')

axes[1].plot(20 * np.sin(2 * np.pi * t_arr / 12), color='green', linewidth=2)
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('Seasonality (Period=12 months)')
axes[1].set_xlabel('Month')

noise_arr = np.random.normal(0, 3, 48)
axes[2].plot(noise_arr, color='red', linewidth=1, alpha=0.7)
axes[2].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[2].set_title('Irregular (Random Noise)')
axes[2].set_xlabel('Time')

plt.suptitle('Individual Time Series Components', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔬 Section 3: Stationarity

**Naye Keywords:**
- **Stationarity** — time series ka mean, variance aur covariance time ke saath constant rahe
- **Non-stationary** — trend ya seasonality wali series (mean change hoti rehti hai)
- **ADF Test (Augmented Dickey-Fuller)** — stationarity test
  - H0: Series non-stationary
  - p-value < 0.05 → Reject H0 → Series IS stationary
- **Differencing** — d(t) = y(t) - y(t-1) — stationarity achieve karne ka tarika
- **d** — ARIMA ka 'd' parameter — kitni baar differencing karni hai

In [ ]:
from statsmodels.tsa.stattools import adfuller

def adf_test(series, name='Series'):
    result = adfuller(series.dropna())
    print(f"ADF Test: {name}")
    print(f"  ADF Statistic : {result[0]:.4f}")
    print(f"  p-value       : {result[1]:.4f}")
    if result[1] < 0.05:
        print("  ✅ Stationary (p < 0.05)")
    else:
        print("  ❌ Non-Stationary (p >= 0.05) — differencing needed")
    print()

# Test original series
adf_test(ts, 'Original Series')

# First difference
ts_diff = ts.diff().dropna()
adf_test(ts_diff, 'First Difference')

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
ts.plot(ax=axes[0], title='Original (Non-Stationary)', color='steelblue')
ts_diff.plot(ax=axes[1], title='After 1st Differencing (Stationary)', color='green')
plt.tight_layout()
plt.show()

---
## 📉 Section 4: ACF aur PACF

**Naye Keywords:**
- **ACF (AutoCorrelation Function)** — series aur uske lagged version ke beech correlation
  - ACF plot se MA(q) order dhundha jaata hai
- **PACF (Partial AutoCorrelation Function)** — sirf direct lag effect (others remove karke)
  - PACF plot se AR(p) order dhundha jaata hai
- **Confidence bands (blue shaded)** — iske bahar ke lags significant hain

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Original series ACF/PACF
plot_acf(ts, lags=30, ax=axes[0, 0], title='ACF — Original')
plot_pacf(ts, lags=30, ax=axes[0, 1], title='PACF — Original', method='ywm')

# After differencing
plot_acf(ts_diff, lags=30, ax=axes[1, 0], title='ACF — After Differencing')
plot_pacf(ts_diff, lags=30, ax=axes[1, 1], title='PACF — After Differencing', method='ywm')

plt.suptitle('ACF and PACF Plots', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Reading ACF/PACF:")
print("  ACF cuts off at lag q → MA(q) model")
print("  PACF cuts off at lag p → AR(p) model")
print("  Both tail off → ARMA(p, q) model")

---
## 🔄 Section 5: AR, MA, ARMA Models

**AR(p) — AutoRegressive Model:**
$$y_t = c + \phi_1 y_{t-1} + \phi_2 y_{t-2} + ... + \phi_p y_{t-p} + \epsilon_t$$
Pichle p values se predict karo.

**MA(q) — Moving Average Model:**
$$y_t = \mu + \epsilon_t + \theta_1 \epsilon_{t-1} + ... + \theta_q \epsilon_{t-q}$$
Pichle q error terms se predict karo.

**ARMA(p,q)** — AR + MA combined

In [ ]:
# --- Generate AR(1) process manually ---
np.random.seed(42)
n = 100
phi = 0.8  # AR coefficient

ar1 = np.zeros(n)
noise = np.random.normal(0, 1, n)
ar1[0] = noise[0]
for t in range(1, n):
    ar1[t] = phi * ar1[t-1] + noise[t]  # AR(1) formula

# Generate MA(1) process
theta = 0.6
ma1 = noise.copy()
for t in range(1, n):
    ma1[t] = noise[t] + theta * noise[t-1]  # MA(1) formula

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ar1, color='steelblue', linewidth=1.5)
axes[0].set_title(f'AR(1) Process: y_t = {phi}*y_(t-1) + noise')

axes[1].plot(ma1, color='darkorange', linewidth=1.5)
axes[1].set_title(f'MA(1) Process: y_t = noise_t + {theta}*noise_(t-1)')

plt.tight_layout()
plt.show()

---
## 📐 Section 6: ARIMA aur SARIMA

**ARIMA(p, d, q):**
- **p** = AR order (PACF se)
- **d** = differencing order (stationarity ke liye)
- **q** = MA order (ACF se)

**SARIMA(p, d, q)(P, D, Q, s):**
- Same as ARIMA + seasonal component
- **s** = seasonality period (12 for monthly, 7 for daily)
- **P, D, Q** = seasonal AR, differencing, MA terms

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

# Use simple stationary series for ARIMA demo
simple_ts = pd.Series(
    100 + np.cumsum(np.random.normal(0.5, 2, 100)) + 10*np.sin(2*np.pi*np.arange(100)/12),
    index=pd.date_range('2015-01', periods=100, freq='MS')
)

train = simple_ts[:80]
test  = simple_ts[80:]

# ARIMA(1,1,1)
model = ARIMA(train, order=(1, 1, 1))
result = model.fit()

print(result.summary().tables[1])

In [ ]:
# Forecast
forecast = result.forecast(steps=20)
conf_int = result.get_forecast(steps=20).conf_int()

plt.figure(figsize=(12, 4))
train.plot(label='Training', color='steelblue')
test.plot(label='Actual Test', color='green')
forecast.plot(label='ARIMA(1,1,1) Forecast', color='red', linestyle='--')
plt.fill_between(conf_int.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1],
                 alpha=0.2, color='red', label='95% CI')
plt.title('ARIMA(1,1,1) Forecast', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

from sklearn.metrics import mean_absolute_error, mean_squared_error
mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))
print(f"MAE  = {mae:.2f}")
print(f"RMSE = {rmse:.2f}")

In [ ]:
# --- SARIMA demo (seasonal) ---
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    sarima_model = SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12),
                           enforce_stationarity=False, enforce_invertibility=False)
    sarima_result = sarima_model.fit(disp=False)
    sarima_forecast = sarima_result.forecast(steps=20)

    plt.figure(figsize=(12, 4))
    train.plot(label='Train', color='steelblue')
    test.plot(label='Test', color='green')
    sarima_forecast.plot(label='SARIMA(1,1,1)(1,1,1,12)', color='purple', linestyle='--')
    plt.title('SARIMA Forecast', fontweight='bold')
    plt.legend()
    plt.tight_layout()
    plt.show()
    print("SARIMA MAE:", mean_absolute_error(test, sarima_forecast).round(2))
except Exception as e:
    print("SARIMA note:", e)

---
## 🏆 Capstone + Solutions

In [ ]:
# Generate website traffic data
np.random.seed(123)
n_traffic = 90
traffic = pd.Series(
    1000 + 2*np.arange(n_traffic) + 150*np.sin(2*np.pi*np.arange(n_traffic)/7) + np.random.normal(0, 30, n_traffic),
    index=pd.date_range('2024-01-01', periods=n_traffic, freq='D'),
    name='Daily_Visits'
)

print("Website Traffic Data:")
print(traffic.head())

# TODO A: Plot the time series
# TODO B: Decompose it (period=7 for weekly seasonality)
# TODO C: ADF test pe run karo
# TODO D: First difference lo, ADF test dobara karo
# TODO E: ARIMA(1,1,1) fit karo on first 75 points, forecast 15 days
# TODO F: MAE aur RMSE calculate karo

print("TODO A-F complete karo!")

In [ ]:
# ===== SOLUTIONS =====
# SOLUTION A
plt.figure(figsize=(12, 3))
traffic.plot(color='steelblue')
plt.title('Daily Website Traffic')
plt.tight_layout(); plt.show()

# SOLUTION B
decomp_t = seasonal_decompose(traffic, model='additive', period=7)
decomp_t.plot()
plt.suptitle('Traffic Decomposition (period=7)', y=1.02)
plt.tight_layout(); plt.show()

# SOLUTION C
adf_test(traffic, 'Traffic')

# SOLUTION D
adf_test(traffic.diff().dropna(), 'Traffic (Differenced)')

# SOLUTION E & F
tr_t, te_t = traffic[:75], traffic[75:]
model_t = ARIMA(tr_t, order=(1, 1, 1)).fit()
fc_t = model_t.forecast(steps=15)
print("MAE :", mean_absolute_error(te_t, fc_t).round(2))
print("RMSE:", np.sqrt(mean_squared_error(te_t, fc_t)).round(2))

plt.figure(figsize=(10, 3))
tr_t.plot(label='Train', color='steelblue')
te_t.plot(label='Actual', color='green')
fc_t.plot(label='Forecast', color='red', linestyle='--')
plt.legend(); plt.tight_layout(); plt.show()

---
## 🎓 Summary

| Concept | Matlab | Use Case |
|---|---|---|
| Trend | Long-term direction | Economic growth, population |
| Seasonality | Fixed periodic pattern | Sales, weather |
| ADF Test | Stationarity check | Before any ARIMA |
| Differencing | Make series stationary | ARIMA preprocessing |
| ACF/PACF | Find AR/MA orders | ARIMA parameter selection |
| ARIMA(p,d,q) | General forecasting | Sales, demand, prices |
| SARIMA | With seasonality | Monthly/yearly patterns |